# BÀI TOÁN 1: PHÂN LOẠI VĂN BẢN SỬ DỤNG NAÏVE BAYES

## 1. Import thư viện và nạp dữ liệu
Bước này bao gồm việc import các thư viện cần thiết như pandas, numpy, sklearn và nạp dữ liệu từ file deceptive-opinion.csv .

In [1]:
# Importing the Necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 1. Nạp dữ liệu vào bộ nhớ
# Thay thế 'spam.csv' trong bài mẫu bằng file dữ liệu mới 'deceptive-opinion.csv'
data = pd.read_csv('deceptive-opinion.csv')

# Hiển thị 5 dòng đầu tiên để kiểm tra cấu trúc dữ liệu
print(data.head())

  deceptive   hotel  polarity       source  \
0  truthful  conrad  positive  TripAdvisor   
1  truthful   hyatt  positive  TripAdvisor   
2  truthful   hyatt  positive  TripAdvisor   
3  truthful    omni  positive  TripAdvisor   
4  truthful   hyatt  positive  TripAdvisor   

                                                text  
0  We stayed for a one night getaway with family ...  
1  Triple A rate with upgrade to view room was le...  
2  This comes a little late as I'm finally catchi...  
3  The Omni Chicago really delivers on all fronts...  
4  I asked for a high floor away from the elevato...  


## 2. Xử lý dữ liệu trước khi xây dựng mô hình
Trong bài mẫu gốc, bước này loại bỏ các cột rác và đổi tên cột. Đối với dữ liệu deceptive-opinion.csv, chúng ta sẽ giữ lại cột nhãn phân loại (deceptive) và cột nội dung (text), đồng thời loại bỏ các cột thông tin phụ (như hotel, polarity, source) để phù hợp với định dạng label và text của bài mẫu.

In [2]:
# 2. Xử lý dữ liệu
# Trong bài mẫu gốc: data.drop(columns=['Unnamed: 2', ...], axis=1)
# Với dữ liệu này, ta loại bỏ các cột không dùng cho việc phân loại văn bản
# Giả sử các cột phụ là 'hotel', 'polarity', 'source' (tùy thuộc vào cấu trúc file thực tế)
cols_to_drop = [col for col in data.columns if col not in ['deceptive', 'text']]
data = data.drop(columns=cols_to_drop, axis=1)

# Đổi tên cột cho giống bài mẫu để dễ theo dõi (label: nhãn, text: nội dung)
# Cột 'deceptive' sẽ đóng vai trò là 'label' (như cột 'v1'/'label' trong bài spam)
data.columns = ['label', 'text']

# Tách đặc trưng (Features - X) và nhãn (Target - y) [cite: 637-638]
X = data.drop('label', axis=1)
y = data['label']

# Chia dữ liệu thành tập huấn luyện và tập kiểm tra (80% training, 20% testing) [cite: 639-640]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Kích thước tập huấn luyện:", X_train.shape)
print("Kích thước tập kiểm tra:", X_test.shape)

Kích thước tập huấn luyện: (1280, 1)
Kích thước tập kiểm tra: (320, 1)


## 3. Vector hóa nội dung (Vectorization)
Chuyển đổi dữ liệu văn bản thành các vector số học sử dụng CountVectorizer. Bước này đếm tần suất xuất hiện của các từ trong văn bản để mô hình có thể hiểu được .

In [3]:
# Khởi tạo CountVectorizer
vectorizer = CountVectorizer()

# Fit và transform dữ liệu huấn luyện (X_train)
# Lưu ý: Truy cập vào cột 'text' của DataFrame X_train như bài mẫu
X_train_vectorized = vectorizer.fit_transform(X_train['text'])

# Transform dữ liệu kiểm tra (X_test)
# Chỉ dùng transform, không fit lại trên tập test để đảm bảo tính nhất quán
X_test_vectorized = vectorizer.transform(X_test['text'])

## 4. Xây dựng mô hình Naïve Bayes
Sử dụng lớp MultinomialNB (phù hợp cho dữ liệu dạng đếm từ) để huấn luyện mô hình trên tập dữ liệu đã được vector hóa .

In [4]:
# Khởi tạo mô hình Multinomial Naive Bayes
classifier = MultinomialNB()

# Huấn luyện mô hình
classifier.fit(X_train_vectorized, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


## 5. Đánh giá hiệu quả của mô hình
Sử dụng mô hình đã huấn luyện để dự đoán trên tập kiểm tra và đánh giá kết quả thông qua độ chính xác (Accuracy), ma trận nhầm lẫn (Confusion Matrix) và báo cáo phân loại (Classification Report) .

In [5]:
# Thực hiện dự đoán trên tập dữ liệu kiểm tra
y_pred = classifier.predict(X_test_vectorized)

# Đánh giá mô hình
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)

# In kết quả
print(f"Accuracy: {accuracy:.2f}")
print("\nConfusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(classification_rep)

Accuracy: 0.87

Confusion Matrix:
[[139  13]
 [ 29 139]]

Classification Report:
              precision    recall  f1-score   support

   deceptive       0.83      0.91      0.87       152
    truthful       0.91      0.83      0.87       168

    accuracy                           0.87       320
   macro avg       0.87      0.87      0.87       320
weighted avg       0.87      0.87      0.87       320

